# Experiment: Exploración del corpus Doppler

**Pregunta.** ¿Cómo están distribuidos clases, duraciones, sample rates y cómo se ve una sirena frente a tráfico en el dominio tiempo-frecuencia?

**Criterio de éxito.** Tablas de balance, histogramas y al menos un ejemplo de waveform + STFT por clase disponible, guardados en `reports/figures/`.


In [10]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

SEED = 7

# Local: data/raw/<corpus>
# Kaggle (Add Input clásico): /kaggle/input/<slug>
# Kaggle (datasets/user): /kaggle/input/datasets/<user>/<slug>/<slug>
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input")
    FIGURES_DIR = Path("/kaggle/working/reports/figures")
    TABLES_DIR = Path("/kaggle/working/reports/tables")
else:
    here = Path.cwd().resolve()
    REPO_ROOT = here if (here / "data" / "raw").exists() else here.parent
    DATA_ROOT = REPO_ROOT / "data" / "raw"
    FIGURES_DIR = REPO_ROOT / "reports" / "figures"
    TABLES_DIR = REPO_ROOT / "reports" / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_SLUGS = {
    "sirennet": ("sirennet",),
    "lssiren": ("lssiren",),
    "urbansound8k": ("urbansound8k",),
}


def is_kaggle() -> bool:
    return IS_KAGGLE


def figures_dir() -> Path:
    return FIGURES_DIR


def tables_dir() -> Path:
    return TABLES_DIR


def resolve_corpus(name: str) -> Path | None:
    candidates: list[Path] = []
    for slug in CORPUS_SLUGS[name]:
        candidates.append(DATA_ROOT / slug)
        datasets = DATA_ROOT / "datasets"
        if datasets.exists():
            for user_dir in datasets.iterdir():
                if not user_dir.is_dir():
                    continue
                candidates.append(user_dir / slug)
                candidates.append(user_dir / slug / slug)
    existing = [path for path in candidates if path.exists()]
    if not existing:
        return None
    return max(existing, key=lambda path: len(path.parts))


@dataclass(frozen=True)
class CorpusPaths:
    sirennet: Path | None
    lssiren: Path | None
    urbansound8k: Path | None

    def available(self) -> dict[str, Path]:
        found = {
            "sirennet": self.sirennet,
            "lssiren": self.lssiren,
            "urbansound8k": self.urbansound8k,
        }
        return {key: path for key, path in found.items() if path is not None}


def corpus_paths() -> CorpusPaths:
    return CorpusPaths(
        sirennet=resolve_corpus("sirennet"),
        lssiren=resolve_corpus("lssiren"),
        urbansound8k=resolve_corpus("urbansound8k"),
    )


print("kaggle:", IS_KAGGLE)
print("DATA_ROOT:", DATA_ROOT)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("available:", list(corpus_paths().available()))
SEED


kaggle: False
DATA_ROOT: /home/jeancdevx/dev/doppler/doppler-ml/data/raw
FIGURES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/figures
TABLES_DIR: /home/jeancdevx/dev/doppler/doppler-ml/reports/tables
available: ['sirennet', 'lssiren', 'urbansound8k']


7

In [13]:
# Inventario de archivos
"""Inventario de archivos de audio del corpus Doppler."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

SIRENNET_CLASS_MAP = {
    "ambulance": "ambulance",
    "police": "police",
    "firetruck": "firetruck",
    "fire_truck": "firetruck",
    "fire": "firetruck",
    "traffic": "traffic",
}

LSSIREN_POSITIVE_HINTS = ("emergency", "siren", "ambulance")
LSSIREN_NEGATIVE_HINTS = ("road", "noise", "traffic")
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}


def is_audio(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_SUFFIXES


def infer_sirennet_label(path: Path) -> str | None:
    parts = [p.lower() for p in path.parts]
    stem = path.stem.lower()
    for key, label in SIRENNET_CLASS_MAP.items():
        if key in parts or stem.startswith(key) or f"_{key}_" in f"_{stem}_":
            return label
    return None


def infer_lssiren_label(path: Path) -> str | None:
    blob = " ".join(p.lower() for p in path.parts)
    if any(h in blob for h in LSSIREN_POSITIVE_HINTS) and "road" not in blob:
        return "siren"
    if any(h in blob for h in LSSIREN_NEGATIVE_HINTS):
        return "road_noise"
    return None


def scan_sirennet(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "sirennet",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": infer_sirennet_label(path) or "unknown",
                "task": "multiclass",
            }
        )
    return pd.DataFrame(rows)


LSSIREN_CSV_COLUMNS = [
    "filename",
    "chroma_stft",
    "rmse",
    "spectral_centroid",
    "spectral_bandwidth",
    "rolloff",
    "zero_crossing_rate",
    *[f"mfcc{i}" for i in range(1, 21)],
    "label",
]


def read_lssiren_features(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    if "filename" in raw.columns:
        return raw
    return pd.read_csv(path, header=None, names=LSSIREN_CSV_COLUMNS)


def scan_lssiren(root: Path) -> pd.DataFrame:
    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        label = infer_lssiren_label(path) or "unknown"
        rows.append(
            {
                "corpus": "lssiren",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": label,
                "task": "binary",
            }
        )
    if rows:
        return pd.DataFrame(rows)

    for csv_path in root.glob("*.csv"):
        feat = read_lssiren_features(csv_path)
        label_col = "label" if "label" in feat.columns else feat.columns[-1]
        name_col = "filename" if "filename" in feat.columns else feat.columns[0]
        for _, row in feat.iterrows():
            raw_label = str(row[label_col]).strip().lower()
            label = "siren" if raw_label in {"ambulance", "siren", "emergency"} else "road_noise"
            rows.append(
                {
                    "corpus": "lssiren",
                    "path": str(csv_path.parent / str(row[name_col])),
                    "relpath": str(row[name_col]),
                    "label": label,
                    "task": "binary",
                    "source": "feature_csv",
                }
            )
    return pd.DataFrame(rows)


def scan_urbansound8k(root: Path) -> pd.DataFrame:
    csv_candidates = list(root.rglob("UrbanSound8K.csv"))
    if csv_candidates:
        meta = pd.read_csv(csv_candidates[0])
        if "class" not in meta.columns and "class_name" in meta.columns:
            meta = meta.rename(columns={"class_name": "class"})
        audio_root = csv_candidates[0].parent.parent / "audio"
        if not audio_root.exists():
            audio_root = root / "audio"
            if not audio_root.exists():
                audio_root = root
        rows = []
        for _, row in meta.iterrows():
            fold = int(row["fold"])
            fname = row["slice_file_name"]
            path = audio_root / f"fold{fold}" / fname
            rows.append(
                {
                    "corpus": "urbansound8k",
                    "path": str(path),
                    "relpath": f"fold{fold}/{fname}",
                    "label": row["class"],
                    "task": "urban_scene",
                    "fold": fold,
                    "fsID": row.get("fsID"),
                    "classID": row.get("classID"),
                    "salience": row.get("salience"),
                }
            )
        return pd.DataFrame(rows)

    rows = []
    for path in root.rglob("*"):
        if not path.is_file() or not is_audio(path):
            continue
        rows.append(
            {
                "corpus": "urbansound8k",
                "path": str(path),
                "relpath": str(path.relative_to(root)),
                "label": path.parent.name,
                "task": "urban_scene",
            }
        )
    return pd.DataFrame(rows)


In [14]:
# Lectura de audio y descriptores
"""Lectura de audio y características con librosa, o scipy si no está disponible."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

try:
    import librosa

    HAS_LIBROSA = True
except Exception:  # pragma: no cover - entorno sin ruedas de librosa
    HAS_LIBROSA = False
    librosa = None  # type: ignore

from scipy.io import wavfile
from scipy.signal import get_window, spectrogram, stft

try:
    import soundfile as sf

    HAS_SOUNDFILE = True
except Exception:
    HAS_SOUNDFILE = False
    sf = None  # type: ignore


TARGET_SR = 22050


@dataclass
class AudioClip:
    y: np.ndarray
    sr: int


def load_audio(path: str, sr: int = TARGET_SR, duration: float | None = None) -> AudioClip:
    if HAS_LIBROSA:
        y, out_sr = librosa.load(path, sr=sr, mono=True, duration=duration)
        return AudioClip(y=np.asarray(y, dtype=np.float32), sr=out_sr)

    if HAS_SOUNDFILE:
        y, file_sr = sf.read(path, always_2d=False)
        y = np.asarray(y, dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if duration is not None:
            y = y[: int(duration * file_sr)]
        if sr and file_sr != sr:
            duration_s = len(y) / float(file_sr)
            n_out = max(1, int(duration_s * sr))
            x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
            x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
            y = np.interp(x_new, x_old, y).astype(np.float32)
            file_sr = sr
        return AudioClip(y=y, sr=int(file_sr))

    file_sr, data = wavfile.read(path)
    y = np.asarray(data, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    max_abs = np.max(np.abs(y)) or 1.0
    if max_abs > 1.5:
        y = y / 32768.0
    if duration is not None:
        y = y[: int(duration * file_sr)]
    if sr and file_sr != sr:
        duration_s = len(y) / file_sr
        n_out = int(duration_s * sr)
        x_old = np.linspace(0.0, duration_s, num=len(y), endpoint=False)
        x_new = np.linspace(0.0, duration_s, num=n_out, endpoint=False)
        y = np.interp(x_new, x_old, y).astype(np.float32)
        file_sr = sr
    return AudioClip(y=y, sr=file_sr)


def duration_seconds(path: str) -> float:
    return float(wav_probe(path)["duration_s"])


def wav_probe(path: str) -> dict:
    """Metadatos baratos sin resamplear."""
    if HAS_SOUNDFILE:
        info = sf.info(path)
        return {
            "sr": int(info.samplerate),
            "n_channels": int(info.channels),
            "n_samples": int(info.frames),
            "duration_s": float(info.duration),
            "dtype": str(info.subtype),
        }
    try:
        sr, data = wavfile.read(path)
        n_channels = 1 if np.asarray(data).ndim == 1 else np.asarray(data).shape[1]
        n_samples = int(np.asarray(data).shape[0])
        return {
            "sr": int(sr),
            "n_channels": int(n_channels),
            "n_samples": n_samples,
            "duration_s": n_samples / float(sr),
            "dtype": str(np.asarray(data).dtype),
        }
    except Exception:
        if HAS_LIBROSA:
            y, sr = librosa.load(path, sr=None, mono=False)
            y = np.asarray(y)
            n_channels = 1 if y.ndim == 1 else y.shape[0]
            n_samples = y.shape[-1]
            return {
                "sr": int(sr),
                "n_channels": int(n_channels),
                "n_samples": int(n_samples),
                "duration_s": n_samples / float(sr),
                "dtype": str(y.dtype),
            }
        raise


def log_mel_spectrogram(clip: AudioClip, n_mels: int = 64, n_fft: int = 1024, hop: int = 256) -> np.ndarray:
    if HAS_LIBROSA:
        S = librosa.feature.melspectrogram(y=clip.y, sr=clip.sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop)
        return librosa.power_to_db(S, ref=np.max)
    f, t, Sxx = spectrogram(clip.y, fs=clip.sr, nperseg=n_fft, noverlap=n_fft - hop, window="hann")
    # Aproximación: filtro triangular en Hz de Mel.
    mel_f = _hz_to_mel(f)
    edges = np.linspace(mel_f.min(), mel_f.max(), n_mels + 2)
    mels = np.zeros((n_mels, Sxx.shape[1]), dtype=np.float32)
    for i in range(n_mels):
        lo, mid, hi = edges[i], edges[i + 1], edges[i + 2]
        w = np.zeros_like(mel_f)
        left = np.logical_and(mel_f >= lo, mel_f <= mid)
        right = np.logical_and(mel_f >= mid, mel_f <= hi)
        if np.any(left):
            w[left] = (mel_f[left] - lo) / max(mid - lo, 1e-8)
        if np.any(right):
            w[right] = (hi - mel_f[right]) / max(hi - mid, 1e-8)
        mels[i] = w @ Sxx
    mels = np.maximum(mels, 1e-10)
    return 10.0 * np.log10(mels / np.max(mels))


def mfcc(clip: AudioClip, n_mfcc: int = 13) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.mfcc(y=clip.y, sr=clip.sr, n_mfcc=n_mfcc)
    log_mel = log_mel_spectrogram(clip, n_mels=40)
    # DCT tipo II sobre el eje mel.
    n_mels, n_frames = log_mel.shape
    n = np.arange(n_mels)
    k = np.arange(n_mfcc)[:, None]
    dct = np.cos(np.pi * k * (2 * n + 1) / (2.0 * n_mels))
    return dct @ log_mel


def spectral_centroid(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_centroid(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    return (f[:, None] * mag).sum(axis=0) / denom


def spectral_bandwidth(clip: AudioClip) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_bandwidth(y=clip.y, sr=clip.sr)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    denom = np.sum(mag, axis=0) + 1e-10
    centroid = (f[:, None] * mag).sum(axis=0) / denom
    var = ((f[:, None] - centroid) ** 2 * mag).sum(axis=0) / denom
    return np.sqrt(var)


def spectral_rolloff(clip: AudioClip, roll_percent: float = 0.85) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.spectral_rolloff(y=clip.y, sr=clip.sr, roll_percent=roll_percent)[0]
    f, _, Zxx = stft(clip.y, fs=clip.sr, nperseg=1024)
    mag = np.abs(Zxx)
    csum = np.cumsum(mag, axis=0)
    thresh = roll_percent * (csum[-1] + 1e-10)
    idx = np.argmax(csum >= thresh, axis=0)
    return f[idx]


def zero_crossing_rate(clip: AudioClip, frame_length: int = 2048, hop: int = 512) -> np.ndarray:
    if HAS_LIBROSA:
        return librosa.feature.zero_crossing_rate(clip.y, frame_length=frame_length, hop_length=hop)[0]
    y = clip.y
    n = 1 + max(0, (len(y) - frame_length) // hop)
    out = np.zeros(n, dtype=np.float32)
    for i in range(n):
        frame = y[i * hop : i * hop + frame_length]
        out[i] = np.mean(np.abs(np.diff(np.signbit(frame))))
    return out


def _hz_to_mel(hz: np.ndarray) -> np.ndarray:
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def stft_db(clip: AudioClip, n_fft: int = 1024, hop: int = 256) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if HAS_LIBROSA:
        S = np.abs(librosa.stft(clip.y, n_fft=n_fft, hop_length=hop))
        db = librosa.amplitude_to_db(S, ref=np.max)
        freqs = librosa.fft_frequencies(sr=clip.sr, n_fft=n_fft)
        times = librosa.frames_to_time(np.arange(db.shape[1]), sr=clip.sr, hop_length=hop)
        return freqs, times, db
    window = get_window("hann", n_fft)
    f, t, Zxx = stft(clip.y, fs=clip.sr, window=window, nperseg=n_fft, noverlap=n_fft - hop)
    mag = np.abs(Zxx)
    db = 20.0 * np.log10(np.maximum(mag, 1e-10) / (np.max(mag) + 1e-10))
    return f, t, db


## Plan

- Hipótesis: sireNNet está casi balanceado en 4 clases; LSSiren es 50/50 binario; UrbanSound8K está desbalanceado y `siren` no distingue tipo.
- Variables: `label`, `duration_s`, `sr`, `n_channels`.
- Métricas: conteos, medianas, % mono, nº de `fsID` únicos vs slices.


In [15]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(SEED)
FIG = figures_dir()
TAB = tables_dir()
plt.rcParams["figure.dpi"] = 120

inv_path = TAB / "file_inventory.csv"
if inv_path.exists():
    inventory = pd.read_csv(inv_path)
else:
    paths = corpus_paths()
    frames = []
    if paths.sirennet:
        frames.append(scan_sirennet(paths.sirennet))
    if paths.lssiren:
        frames.append(scan_lssiren(paths.lssiren))
    if paths.urbansound8k:
        frames.append(scan_urbansound8k(paths.urbansound8k))
    inventory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not inventory.empty:
        inventory.to_csv(inv_path, index=False)

print(inventory.groupby(["corpus", "label"]).size() if not inventory.empty else "inventory empty")
inventory.head()


corpus        label           
lssiren       road_noise           902
              siren                932
sirennet      ambulance            400
              firetruck            400
              police               454
              traffic              421
urbansound8k  air_conditioner     1000
              car_horn             429
              children_playing    1000
              dog_bark            1000
              drilling            1000
              engine_idling       1000
              gun_shot             374
              jackhammer          1000
              siren                929
              street_music        1000
dtype: int64


,corpus,path,relpath,label,task,source,fold,fsID,classID,salience
0,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_658.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
1,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_647.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
2,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_791_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
3,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_785_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
4,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_685_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN


## Balance de clases


In [16]:
if inventory.empty:
    print("No hay inventario.")
    counts = pd.DataFrame(columns=["corpus", "label", "n"])
else:
    counts = inventory.groupby(["corpus", "label"]).size().reset_index(name="n")
    n_corpus = max(int(counts["corpus"].nunique()), 1)
    fig, axes = plt.subplots(1, n_corpus, figsize=(4.2 * n_corpus, 4), squeeze=False)
    for ax, (corpus, sub) in zip(axes[0], counts.groupby("corpus")):
        sns.barplot(data=sub.sort_values("n", ascending=False), x="label", y="n", ax=ax, color="#3b6d9a")
        ax.set_title(corpus)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(FIG / "class_balance.png", bbox_inches="tight")
    plt.show()
counts


/tmp/ipykernel_17792/2174148442.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,corpus,label,n
0,lssiren,road_noise,902
1,lssiren,siren,932
2,sirennet,ambulance,400
3,sirennet,firetruck,400
4,sirennet,police,454
5,sirennet,traffic,421
6,urbansound8k,air_conditioner,1000
7,urbansound8k,car_horn,429
8,urbansound8k,children_playing,1000
9,urbansound8k,dog_bark,1000


## Duración, sample rate y canales

Se sondean los WAV existentes. Si UrbanSound8K o LSSiren están solo como CSV, esas filas se omiten del probe.


In [17]:
from pathlib import Path as _P

audio_rows = inventory.copy()
audio_rows = audio_rows[audio_rows["path"].map(lambda p: _P(str(p)).exists() and _P(str(p)).suffix.lower() in {".wav", ".mp3", ".flac"})]

def probe_row(path: str) -> dict:
    try:
        return wav_probe(path)
    except Exception as exc:
        return {"sr": None, "n_channels": None, "n_samples": None, "duration_s": None, "dtype": str(exc)}

probes = []
for _, row in audio_rows.iterrows():
    meta = probe_row(row["path"])
    meta.update({"corpus": row["corpus"], "label": row["label"], "path": row["path"]})
    probes.append(meta)

probe_df = pd.DataFrame(probes)
if not probe_df.empty:
    probe_df.to_csv(TAB / "wav_probe.csv", index=False)
    summary = probe_df.groupby(["corpus", "label"]).agg(
        n=("duration_s", "count"),
        duration_median=("duration_s", "median"),
        duration_mean=("duration_s", "mean"),
        sr_median=("sr", "median"),
        channels_mean=("n_channels", "mean"),
    ).reset_index()
    summary.to_csv(TAB / "wav_probe_summary.csv", index=False)
else:
    summary = pd.DataFrame()
summary


,corpus,label,n,duration_median,duration_mean,sr_median,channels_mean
0,sirennet,ambulance,400,3.000091,3.001898,44100.0,2.0
1,sirennet,firetruck,400,3.000091,3.002029,44100.0,2.0
2,sirennet,police,454,3.000045,3.000045,44100.0,2.0
3,sirennet,traffic,421,3.000000,3.000000,44100.0,2.0


In [18]:
if not probe_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.histplot(data=probe_df, x="duration_s", hue="corpus", bins=40, ax=ax, element="step")
    ax.set_xlabel("Duración (s)")
    fig.tight_layout()
    fig.savefig(FIG / "duration_hist.png", bbox_inches="tight")
    plt.show()
else:
    print("Sin WAV para histograma de duración")


/tmp/ipykernel_17792/2632699863.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## UrbanSound8K: slices vs recordings

Aunque no haya audio, el CSV oficial permite medir el riesgo de data leakage por `fsID`.


In [19]:

us_root = resolve_corpus("urbansound8k")
us_csv = None
if us_root:
    hits = list(us_root.rglob("UrbanSound8K.csv"))
    us_csv = hits[0] if hits else None

if us_csv is None:
    print("UrbanSound8K.csv no montado")
    us_meta = pd.DataFrame()
else:
    us_meta = pd.read_csv(us_csv)
    if "class" not in us_meta.columns and "class_name" in us_meta.columns:
        us_meta = us_meta.rename(columns={"class_name": "class"})
    leakage = us_meta.groupby("class").agg(n_slices=("slice_file_name", "nunique"), n_recordings=("fsID", "nunique")).reset_index()
    leakage["slices_per_recording"] = leakage["n_slices"] / leakage["n_recordings"]
    leakage.to_csv(TAB / "urbansound8k_slices_vs_recordings.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=us_meta["class"].value_counts().rename_axis("class").reset_index(name="n"), x="class", y="n", ax=ax, color="#6b4c9a")
    ax.tick_params(axis="x", rotation=45)
    ax.set_title("UrbanSound8K — clips por clase")
    fig.tight_layout()
    fig.savefig(FIG / "urbansound8k_class_counts.png", bbox_inches="tight")
    plt.show()
    display_df = leakage
    display_df


/tmp/ipykernel_17792/4251169687.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Formas de onda y STFT por clase (sireNNet)

Un ejemplo aleatorio por etiqueta, misma escala temporal.


In [20]:

sirennet = audio_rows[audio_rows["corpus"] == "sirennet"] if not audio_rows.empty else pd.DataFrame()
if sirennet.empty:
    print("sireNNet WAV no montado: se omite waveform/STFT")
else:
    examples = []
    for label, sub in sirennet.groupby("label"):
        examples.append(sub.sample(1, random_state=SEED).iloc[0])
    n = len(examples)
    fig, axes = plt.subplots(n, 2, figsize=(10, 2.4 * n), squeeze=False)
    for i, row in enumerate(examples):
        clip = load_audio(row["path"], duration=3.0)
        t = np.arange(len(clip.y)) / clip.sr
        axes[i, 0].plot(t, clip.y, color="#1f3b57", lw=0.6)
        axes[i, 0].set_ylabel(row["label"])
        axes[i, 0].set_xlim(0, t[-1] if len(t) else 1)
        freqs, times, db = stft_db(clip)
        im = axes[i, 1].pcolormesh(times, freqs, db, shading="auto", cmap="magma")
        axes[i, 1].set_ylim(0, 8000)
        if i == 0:
            axes[i, 0].set_title("Waveform")
            axes[i, 1].set_title("STFT (dB)")
    axes[-1, 0].set_xlabel("t (s)")
    axes[-1, 1].set_xlabel("t (s)")
    fig.tight_layout()
    fig.savefig(FIG / "sirennet_waveform_stft.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_17792/45453273.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultados

- El balance y las duraciones condicionan el padding/windowing del modelo.
- UrbanSound8K no debe partirse al azar: varios slices salen del mismo `fsID`.
- Siguiente: MFCC, log-mel y descriptores espectrales por clase.


In [21]:
result = {
    "seed": SEED,
    "n_inventory": int(len(inventory)),
    "n_wav_probed": int(len(probe_df)) if "probe_df" in globals() else 0,
    "figures": sorted(p.name for p in FIG.glob("*.png")),
}
result


{'seed': 7,
 'n_inventory': 12241,
 'n_wav_probed': 1675,
 'figures': ['class_balance.png',
  'descriptors_by_class.png',
  'duration_hist.png',
  'logmel_vs_mfcc_examples.png',
  'mfcc_means_by_class.png',
  'sirennet_waveform_stft.png',
  'urbansound8k_class_counts.png']}